In [1]:
!pip install ipywidgets
import numpy as np
import pandas as pd
from labeler import LabelingWidget

class FakeEmbedder:
    """Deterministic random embeddings keyed by text hash — for UI testing only."""
    def encode(self, texts, normalize_embeddings=False, **kwargs):
        vecs = np.stack([
            np.random.default_rng(abs(hash(t)) % (2**32)).standard_normal(64)
            for t in texts
        ])
        if normalize_embeddings:
            vecs /= np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12
        return vecs

df = pd.DataFrame({"text": [
    "Love it, fast shipping!",
    "Support never replied.",
    "Too expensive for the quality.",
    "Works fine, nothing special.",
    "Broke after a week.",
]})

label_dict = {
    "sentiment": {"positive": "approval", "negative": "disapproval", "neutral": "factual"},
    "topic":     {"pricing": "cost", "support": "service", "quality": "build quality"},
}

widget = LabelingWidget(
    embed_model=FakeEmbedder(),
    label_dict=label_dict,
    df=df,
    save_path="labeled_test.parquet",
)

You should consider upgrading via the 'C:\Users\jaspe\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from labeler import LabelingWidget

texts = [
    "Absolutely love this product, worth every penny and arrived super fast!",
    "Customer support never got back to me after three emails. Very frustrating.",
    "The price is fair but the build quality feels cheap and flimsy.",
    "Works as described. Nothing special, nothing bad.",
    "Honestly the best purchase I've made this year, fantastic quality.",
    "Way too expensive for what you get. Felt ripped off.",
    "The rep on the phone was incredibly helpful and resolved my issue in minutes.",
    "Stopped working after two weeks. Defective unit.",
    "Shipping took forever but the item itself is fine.",
    "Decent value for the money, no complaints.",
    "Terrible experience from start to finish. Avoid.",
    "Solid product, would buy again.",
]
df = pd.DataFrame({"text": texts})

label_dict = {
    "sentiment": {
        "positive": "expresses approval, happiness, or satisfaction with the product or experience",
        "negative": "expresses disapproval, anger, disappointment, or frustration",
        "neutral":  "factual or descriptive with no clear emotional valence",
    },
    "topic": {
        "pricing":  "mentions cost, price, fees, value for money, or being expensive/cheap",
        "support":  "mentions customer service, help, agents, response time, or communication",
        "quality":  "mentions product quality, defects, durability, materials, or how well it works",
        "shipping": "mentions delivery, shipping speed, packaging, or arrival",
    },
}


from IPython.display import display
from sentence_transformers import SentenceTransformer
from labeler import LabelingWidget

model = SentenceTransformer("all-MiniLM-L6-v2")

widget = LabelingWidget(
    embed_model=model,
    label_dict=label_dict,
    df=df,
    save_path="labeled_demo.csv",
    text_column="text",
)

display(widget)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

LabelingWidget(children=(HTML(value="<h3 style='margin:4px 0'>1 / 12 <span style='color:#888;font-weight:norma…

In [3]:
from active_learner import ActiveLearner
from mlflow_model import SentenceTransformerClassifier

label_dict = {
    "sentiment": {
        "positive": "expresses approval, happiness, or satisfaction",
        "negative": "expresses disapproval, anger, or frustration",
        "neutral":  "factual or descriptive with no clear emotional valence",
    },
    "topic": {
        "pricing":  "mentions cost, price, fees, value for money, expensive or cheap",
        "support":  "mentions customer service, help, agents, response time",
        "quality":  "mentions product quality, defects, durability, materials",
        "shipping": "mentions delivery, shipping speed, packaging, arrival",
    },
}

clf = SentenceTransformerClassifier(
    label_dict=label_dict,
    model_name_or_path="Qwen/Qwen3-Embedding-0.6B",
    threshold=0.45,
    n_trainable_transformer_layers=1,
)

widget = ActiveLearner(
    model=clf,
    labeled_test_path="labeled_demo.csv",
    labeled_train_path="labeled_train_al.csv",
    unlabeled_train_path="unlabeled_train.csv",
    retrain_every=8,
    epochs=1,
    batch_size=8,
    query_strategy="margin",
    model_save_path="finetuned_model",
)
widget

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[ActiveLearner] pool ready: 300 rows total (103 pre-labeled from labeled_train_al.csv, 197 new from unlabeled_train.csv)


ActiveLearner(children=(HTML(value="<h3 style='margin:4px 0'>104 / 300 <span style='color:#888;font-weight:nor…

In [1]:
label_dict = {
    "sentiment": {
        "positive": "expresses approval, happiness, or satisfaction",
        "negative": "expresses disapproval, anger, or frustration",
        "neutral":  "factual or descriptive with no clear emotional valence",
    },
    "topic": {
        "pricing":  "mentions cost, price, fees, value for money, expensive or cheap",
        "support":  "mentions customer service, help, agents, response time",
        "quality":  "mentions product quality, defects, durability, materials",
        "shipping": "mentions delivery, shipping speed, packaging, arrival",
    },
}

In [ ]:
from active_learner import ActiveLearner
from xgboost_model import XGBoostMatcher

clf = XGBoostMatcher(
    label_dict=label_dict,
    embed_model_name_or_path="all-MiniLM-L6-v2",  # frozen encoder
    threshold=0.5,
    use_pca=True,                # PCA before XGBoost
    pca_components=64,
    feature_mode="concat+diff+prod",  # both embeddings + diff + product
    n_negatives_per_text=3,
    n_estimators_per_epoch=100,  # widget.epochs × this = total boost rounds
    max_depth=6,
    learning_rate=0.1,
)

widget = ActiveLearner(
    model=clf,
    labeled_test_path="labeled_demo.csv",
    labeled_train_path="labeled_train_al.csv",
    unlabeled_train_path="unlabeled_train.csv",
    retrain_every=8,
    epochs=1,
    batch_size=500,
    query_strategy="margin",
)
widget


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[ActiveLearner] pool ready: 300 rows total (103 pre-labeled from labeled_train_al.csv, 197 new from unlabeled_train.csv)


ActiveLearner(children=(HTML(value="<h3 style='margin:4px 0'>104 / 300 <span style='color:#888;font-weight:nor…

In [4]:
!pip install -U "torch"

  Using cached torch-2.12.0-cp310-cp310-win_amd64.whl (122.9 MB)
  Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
  Attempting uninstall: sympy
    Found existing installation: sympy 1.13.1
    Uninstalling sympy-1.13.1:
      Successfully uninstalled sympy-1.13.1
  Attempting uninstall: torch
    Found existing installation: torch 2.5.1+cu121
    Uninstalling torch-2.5.1+cu121:
      Successfully uninstalled torch-2.5.1+cu121


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\jaspe\\AppData\\Local\\Programs\\Python\\Python310\\Lib\\site-packages\\~-rch\\lib\\asmjit.dll'
Consider using the `--user` option or check the permissions.

You should consider upgrading via the 'C:\Users\jaspe\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [5]:

widget._eval_df

,text,labels
0,"Box arrived crushed, item scratched and missin...","[sentiment::negative, topic::shipping]"
1,Support rep was patient and walked me through ...,"[sentiment::positive, topic::support]"
2,"Quick chat response, polite agent, problem sol...","[sentiment::positive, topic::support]"
3,Reasonably priced and the materials hold up un...,"[sentiment::positive, topic::pricing, topic::q..."
4,Top-notch finish and the metal frame feels roc...,"[sentiment::positive, topic::quality]"
5,Customer service hung up on me twice and never...,"[sentiment::negative, topic::support]"
6,"Defective, expensive, slow to arrive, and supp...","[sentiment::negative, topic::quality, topic::p..."
7,Delivery driver left the package outside in th...,"[sentiment::negative, topic::shipping]"
8,"Hidden fees everywhere, the real cost is doubl...","[sentiment::negative, topic::pricing]"
9,"Incredible value for the price, can't believe ...","[sentiment::positive, topic::pricing]"
